# Friday Night Frisbee Video Browser

This notebook builds a standalone HTML page with a YouTube player beside a Shown Space-style field. Clicking a throw seeks the video to an estimated timestamp from manually entered point-start anchors.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")

from ufa import (
    add_estimated_video_seconds,
    build_fnf_browser_data,
    build_fnf_game_table,
    load_fnf_point_anchors,
    write_fnf_video_browser_html,
)

## Find Friday Night Frisbee Games

This searches the UFA schedule metadata for Friday games, then joins in manually verified Friday Night Frisbee YouTube URLs from `../data/manual/fnf_games.csv`. The UFA metadata is useful for finding Friday games, but the broadcast URL often points to WatchUFA rather than the free YouTube video.

In [ ]:
FNF_START_DATE = "2026-04-24"
FNF_END_DATE = "2026-08-01"
TEAM_ID = None  # Leave as None for both teams, or set to a team id like "glory".
FNF_GAMES_CSV = Path("../data/manual/fnf_games.csv")

fnf_games = build_fnf_game_table(
    FNF_START_DATE,
    FNF_END_DATE,
    schedule_path=FNF_GAMES_CSV,
    season=2026,
    team_id=TEAM_ID,
)

fnf_games

## Choose A Friday Night Frisbee Game

Pick the row number from `fnf_games` that you want to turn into a browser. Rows where `is_fnf_youtube` is `True` have a usable YouTube URL. If the row you want is `False`, add that game and YouTube URL to `../data/manual/fnf_games.csv`, then rerun the table cell.

In [ ]:
GAME_ROW = 0

if fnf_games.empty:
    raise ValueError("No Friday games were found for this date range.")

selected_game = fnf_games.iloc[GAME_ROW]
GAME_ID = selected_game["gameID"]
YOUTUBE_URL = selected_game["youtube_url"]

if not isinstance(YOUTUBE_URL, str) or not YOUTUBE_URL.strip():
    raise ValueError("This row does not have a YouTube URL yet. Add it to ../data/manual/fnf_games.csv, then rerun the FNF table cell.")

ANCHORS_CSV = Path("../data/manual/fnf_point_anchors.csv")
OUTPUT_HTML = Path(f"../outputs/fnf_browsers/{GAME_ID}.html")

print(GAME_ID)
print(YOUTUBE_URL)
selected_game

## Manual Point Anchors

Add point-start anchors to `../data/manual/fnf_point_anchors.csv`. Each row should say where a game point starts in the YouTube video.

Required columns:

`game_id, youtube_url, game_quarter, quarter_point, video_seconds, note`

In [ ]:
anchors = load_fnf_point_anchors(ANCHORS_CSV, GAME_ID)
anchors

## Fetch Throws And Build Browser Data

In [ ]:
possessions, paths = build_fnf_browser_data(GAME_ID, team_id=TEAM_ID)
paths = add_estimated_video_seconds(paths, anchors, seconds_per_throw=3.0)

print(f"Scoring possessions: {len(possessions):,}")
print(f"Paths: {len(paths):,}")
possessions.head()

## Export HTML Browser

In [ ]:
output_path = write_fnf_video_browser_html(
    game_id=GAME_ID,
    youtube_url=YOUTUBE_URL,
    possessions=possessions,
    paths=paths,
    output_path=OUTPUT_HTML,
)

output_path

Open the generated HTML file in a browser. In the browser, choose a possession, click a throw, and the YouTube player will seek to that estimated moment. Left/Right arrow keys move through throws after focusing the field.